# Task 02 — Silver: Orders Clean & Enrich — **SOLUTION**

**Workshop: Final Pipeline | Layer 2 of 3**

## Goal

Read `bronze.lab_orders`, apply five transformations, write to `silver.lab_orders`.

```
bronze.lab_orders
        |
        v  cast * withColumn * when/otherwise * filter * drop
        v
silver.lab_orders
```

## Transformations to implement

| # | What | Details |
|---|------|--------|
| 1 | `cast` | `order_datetime` String -> TimestampType |
| 2 | `withColumn` | Add `net_amount` = `total_amount` x (1 - `discount_percent` / 100), rounded to 2 dp |
| 3 | `when / otherwise` | Add `order_tier`: `"Premium"` >= 500 * `"Standard"` >= 200 * `"Small"` otherwise |
| 4 | `filter` | Remove rows where `order_id` IS NULL |
| 5 | `drop` | Remove `_source_file` (Bronze metadata, not needed in Silver) |

> `net_amount` is used by Task 03 for revenue — make sure the formula is correct.

In [ ]:
%run ../../setup/00_setup

In [ ]:
dbutils.widgets.text("catalog",       CATALOG,       "Catalog")
dbutils.widgets.text("bronze_schema", BRONZE_SCHEMA, "Bronze Schema")
dbutils.widgets.text("silver_schema", SILVER_SCHEMA, "Silver Schema")

catalog       = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

source_table = f"{catalog}.{bronze_schema}.lab_orders"
target_table = f"{catalog}.{silver_schema}.lab_orders"

print(f"Source : {source_table}")
print(f"Target : {target_table}")

---

## Exercise 1 — Imports

Import all PySpark functions needed for the five transformations above.

In [ ]:
from pyspark.sql.functions import col, to_timestamp, when, round, lit

---

## Exercise 2 — Read the Bronze table

In [ ]:
# Read the Bronze table into orders_bronze (Task 01 must have run first)
orders_bronze = spark.table(source_table)

In [ ]:
print(f"Bronze rows : {orders_bronze.count():,}")
orders_bronze.printSchema()

---

## Exercise 3 — Apply all five transformations in one chain

Chain all transformations into a single expression. Name the result `orders_silver`.

In [ ]:
orders_silver = (
    orders_bronze
    # 1. Cast order_datetime: String → TimestampType
    .withColumn("order_datetime",
        to_timestamp(col("order_datetime"), "yyyy-MM-dd'T'HH:mm:ss"))
    # 2. Add net_amount = total_amount × (1 - discount_percent / 100), rounded to 2 dp
    .withColumn("net_amount",
        round(col("total_amount") * (lit(1) - col("discount_percent") / lit(100)), 2))
    # 3. Add order_tier based on net_amount
    .withColumn("order_tier",
        when(col("net_amount") >= 500, "Premium")
        .when(col("net_amount") >= 200, "Standard")
        .otherwise("Small"))
    # 4. Remove rows where order_id is null
    .filter(col("order_id").isNotNull())
    # 5. Drop Bronze metadata column not needed in Silver
    .drop("_source_file")
)

In [ ]:
print(f"Silver rows : {orders_silver.count():,}")
orders_silver.select(
    "order_id","order_datetime","total_amount",
    "discount_percent","net_amount","order_tier"
).show(10, truncate=False)
orders_silver.printSchema()

---

## Exercise 4 — Write to Silver as a managed Delta table

In [ ]:
(
    orders_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
)

In [ ]:
row_count = spark.table(target_table).count()
print(f"OK  {target_table}  ->  {row_count:,} rows")
display(spark.table(target_table).limit(5))

In [ ]:
import json
dbutils.notebook.exit(json.dumps({"status":"SUCCESS","table":target_table,"rows":row_count}))